# Phase 5 · Train the warehouse object detector (YOLO) on a Colab GPU

**What this does:** trains a YOLO object detector on the 2,400 camera images rendered from our 3D
warehouse (6 classes: robot, worker, shelf, package, obstacle, blocked_path), evaluates it on the
held-out **test** images, and packs the trained model and its reports into one zip to download.

**Before you start**
1. *Runtime → Change runtime type → **T4 GPU** → Save.*
2. Have `warehouse_vision_v3.zip` ready (made on your computer by
   `python scripts/package_vision_dataset.py`, saved in `data/exports/`). Easiest: put it in the top
   folder of your **Google Drive** (upload once, reuse every time). Direct upload also works (step 2, option B).
3. *Runtime → **Run all**.* Training takes about 30-45 minutes. Keep the tab open.

**At the end** download `warehouse_yolo_results.zip` and unzip it into
`H:\ai-warehouse-platform\models\` (you get `models/vision_yolo_v1/`).

Settings are fixed (seed, epochs, image size) so the run is reproducible; ultralytics is pinned to
**8.4.178**, the same version as on your computer.

## 1. Check the GPU and install the detector library

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
%pip install -q ultralytics==8.4.178
import torch, ultralytics
print("torch", torch.__version__, "| CUDA GPU:", torch.cuda.is_available(), "| ultralytics", ultralytics.__version__)
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> T4 GPU, then Run all again"


## 2. Get the dataset

**Option A (recommended):** `warehouse_vision_v3.zip` is in the top folder of your Google Drive;
Colab asks once for permission to read your Drive.
**Option B:** set `USE_DRIVE = False`; a file picker appears, choose the zip (slower, about 265 MB).

In [ ]:
USE_DRIVE = True
ZIP_NAME = "warehouse_vision_v3.zip"

import shutil, zipfile
from pathlib import Path

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    src = Path("/content/drive/MyDrive") / ZIP_NAME
    assert src.exists(), f"{src} not found: put {ZIP_NAME} in the top folder of My Drive (or set USE_DRIVE = False)"
    shutil.copy(src, "/content/" + ZIP_NAME)
else:
    from google.colab import files
    files.upload()

with zipfile.ZipFile("/content/" + ZIP_NAME) as z:
    z.extractall("/content")
DATA = Path("/content/warehouse_vision")
for split in ("train", "validation", "test"):
    print(split, len(list((DATA / "images" / split).glob("*.jpg"))), "images")


## 3. Look at the data

Number of labelled objects per class. Shelves dominate; obstacles and packages are rare, so we
report scores **per class**, not only the average.

In [ ]:
import collections, yaml
names = yaml.safe_load((DATA / "data.yaml").read_text())["names"]
(DATA / "data.yaml").write_text((DATA / "data.yaml").read_text().replace("path: .", f"path: {DATA}"))

for split in ("train", "validation", "test"):
    counts = collections.Counter()
    for f in (DATA / "labels" / split).glob("*.txt"):
        counts.update(int(line.split()[0]) for line in f.read_text().splitlines() if line.strip())
    print(f"{split:<11}", {names[k]: counts[k] for k in sorted(counts)})


## 4. Train

* **Model:** YOLO11s, pretrained on COCO (transfer learning: it already knows edges, shapes and
  objects; we teach it our 6 classes).
* **Image size 960:** robots are only about 11 pixels wide in a 640-pixel image; enlarging helps the
  detector see small objects.
* **Early stopping:** training stops if the validation score has not improved for 20 epochs; the best
  epoch is kept (`best.pt`).
* Selection uses the **validation** images only; the **test** images are used once, in step 5.

In [ ]:
from ultralytics import YOLO

MODEL = "yolo11s.pt"
model = YOLO(MODEL)
train = model.train(
    data=str(DATA / "data.yaml"),
    epochs=100, patience=20, imgsz=960, batch=16,
    seed=42, deterministic=True,
    project="/content/runs", name="vision_yolo_v1", exist_ok=True,
    plots=True,
)
RUN = Path(model.trainer.save_dir)
print("saved to", RUN)


## 5. Evaluate on the held-out test images

* **Precision:** of the boxes the model drew, the share that were right.
* **Recall:** of the real objects, the share the model found.
* **mAP@0.5:** average precision over all classes when a box counts as correct if it overlaps the
  true box by at least 50 % (IoU >= 0.5). **mAP@0.5:0.95** averages stricter overlaps, 50-95 %.

In [ ]:
import json
best = YOLO(str(RUN / "weights" / "best.pt"))
test = best.val(data=str(DATA / "data.yaml"), split="test", imgsz=960, batch=16,
                project="/content/runs", name="vision_yolo_v1_test", exist_ok=True, plots=True)

per_class = {}
for i, c in enumerate(test.box.ap_class_index):
    p, r, ap50, ap = test.box.class_result(i)
    per_class[names[int(c)]] = {"precision": round(float(p), 3), "recall": round(float(r), 3),
                                "mAP50": round(float(ap50), 3), "mAP50_95": round(float(ap), 3)}
report = {
    "model": "vision_yolo_v1", "algorithm": "YOLO11s (Ultralytics), COCO-pretrained, fine-tuned",
    "ultralytics": ultralytics.__version__, "imgsz": 960,
    "epochs_run": sum(1 for _ in open(RUN / "results.csv")) - 1,
    "test": {"precision": round(float(test.box.mp), 3), "recall": round(float(test.box.mr), 3),
             "mAP50": round(float(test.box.map50), 3), "mAP50_95": round(float(test.box.map), 3)},
    "test_per_class": per_class,
    "speed_ms_per_image": {k: round(v, 2) for k, v in test.speed.items()},
    "gpu": torch.cuda.get_device_name(0),
}
print(json.dumps(report, indent=2))


## 6. Pack the results and download

In [ ]:
import shutil
out = Path("/content/vision_yolo_v1")
out.mkdir(exist_ok=True)
run = RUN
shutil.copy(run / "weights" / "best.pt", out / "best.pt")
for f in ["results.csv", "results.png", "args.yaml"]:
    if (run / f).exists():
        shutil.copy(run / f, out / f)
test_dir = Path(test.save_dir)
for f in test_dir.glob("*.png"):
    shutil.copy(f, out / ("test_" + f.name))
(out / "report.json").write_text(json.dumps(report, indent=2))

shutil.make_archive("/content/warehouse_yolo_results", "zip", "/content", "vision_yolo_v1")
print("files:", sorted(p.name for p in out.iterdir()))

from google.colab import files
files.download("/content/warehouse_yolo_results.zip")


**Done.** Unzip `warehouse_yolo_results.zip` into `H:\ai-warehouse-platform\models\` so that
`models/vision_yolo_v1/best.pt` exists, then tell Claude: the evaluation notebook and the Camera view
use it from there. If the download did not start, open the folder icon on the left and download
`warehouse_yolo_results.zip` by hand.